## blah

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from jax import random

# ---- product-space likelihoods (single-point each) ----
# M2: N([0,0,0,0], diag([1,1, 0.3^2, 0.3^2])) over args = [θ1(2), θ2(2)]
def loglik_M2_single(theta_4):                 # (4,)
    # standard normal on first two, tighter on last two just to shape landscape
    return -0.5 * jnp.sum(theta_4[:2]**2) - 0.5 * jnp.sum((theta_4[2:]/0.3)**2) - 2*jnp.log(0.3)

# M3: same on the first 4, plus θ3 ~ N([0,0], diag([0.5^2, 0.5^2]))
def loglik_M3_single(theta_6):                 # (6,)
    base = -0.5*jnp.sum(theta_6[:2]**2) - 0.5*jnp.sum((theta_6[2:4]/0.3)**2) - 2*jnp.log(0.3)
    extra = -0.5*jnp.sum((theta_6[4:]/0.5)**2) - 2*jnp.log(0.5)
    return base + extra

# ψ3 prior (uniform box) for rejuvenating θ3: here use standard normal proxy (log-ψ3)
def psi3_logpdf(theta_3):                      # (2,)
    # treat as 0-mean N with σ=1 (for demo)
    return -0.5*jnp.sum(theta_3**2)

def psi3_sample(key, N, d3):
    # sample N x d3 from N(0,1) (simple & works inside jit)
    return random.normal(key, (N, d3))

# ---- ensemble shapes ----
C = 6               # temperatures
W = 6               # walkers
D = 6               # 3 blocks of size 2
Npar_src = 2
key = random.PRNGKey(0)

# temperatures (geometric)
T = jnp.array([1.0 * (1.7**(c/(C-1))) for c in range(C)])

# initial ensemble around origin
key, k0 = random.split(key)
initial_thetas = 0.05 * random.normal(k0, (C, W, D))

# optional: initialize some walkers in M3 (z=1) to avoid label starvation
z0 = np.zeros((C, W), dtype=np.int32)
z0[:, ::2] = 1   # every other walker starts as M3

base_cov = np.eye(D)

# 4 components (Student-t, eigen-line, fullcov, stretch). You can bias to stretch for product-space.
weights = np.broadcast_to(np.array([0.25, 0.2, 0.2, 0.35]), (C, W, 4))

# adapt config
class AdaptConfig:
    m_epochs = 60
    N_steps = 32
    eta = 0.05
    target_accept = 0.25
    scale_init = 1.0
    kappa_line = 0.5
    scale_min = 0.05
    scale_max = 5.0
    shrink = 0.1
    jitter = 1e-9

cfg = AdaptConfig()

# ----- run (product space) -----
state, out = run_adaptive_pt_device_fast(
    key,
    initial_thetas=initial_thetas,       # (C,W,D)
    temperatures=T,                      # (C,)
    log_prob_fn_single=None,             # ignored in product_space=True
    base_cov=base_cov,
    weights=weights,
    cfg=cfg,
    lik_chunk=64,
    product_space=True,                  # <— product space
    Npar_src=Npar_src,
    loglik_M2_single=loglik_M2_single,
    loglik_M3_single=loglik_M3_single,
    model_update_stride=4,
    log_prior_z=(0.0, 0.0),              # flat label prior
    psi3_sample=psi3_sample,
    psi3_logpdf=psi3_logpdf,
    initial_z=z0
)

print("Final shapes:")
print("  thetas:", state.thetas.shape)       # (C,W,D)
print("  z:", state.z.shape)                  # (C,W)
print("Mean accept per epoch (per chain):", out["accept_rate_per_epoch"][-1])
print("Mean swap rate last epoch:", out["swap_rate_per_epoch"][-1])

# Inspect last epoch’s compact info (product-space)
slim_last = out["slim_per_epoch"][-1]  # SlimInfoPS
print("Accepted M2 points [c=0, w=0]:", np.asarray(slim_last.accepted_points_M2[0][0]).shape)
print("Accepted M3 points [c=0, w=0]:", np.asarray(slim_last.accepted_points_M3[0][0]).shape)
print("z_time_in_M3 (chain 0):", slim_last.z_time_in_M3[0])